# 02 · Feature Engineering

**Question**: which features are worth building for a pre-release revenue model, and can prior-movie history be used without leaking the future?

Decisions made here feed the five feature-stage experiments in `04.model_comparison.ipynb`.

- Prediction point is **before release**, so post-release signals (popularity, votes, revenue itself) are never used as features.
- Historical features use **only movies released strictly before** the film's release date (release-date ties are excluded as concurrent).
- A separate unfiltered history extract (`queries/extract_history_features.sql`) is used so director/company/cast/franchise history is not computed from the already revenue-filtered ML extract.

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "scripts").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))


In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings("ignore")
plt.rcParams["figure.dpi"] = 100
pd.set_option("display.max_columns", None)


In [3]:
from scripts.load_data import load_feature_dataset, load_history_data
from scripts.validate_data import validate_dataset
from scripts.prepare_dataset import prepare_ml_dataset
from scripts.historical_features import compute_historical_features, prepare_history_table, verify_historical_features
import scripts.feature_engineering as fe
import scripts.evaluation as ev

## 1. The ML extract (already revenue-filtered)

In [4]:
df = load_feature_dataset()
df = validate_dataset(df)
df.shape

rows: 8040
columns: 43
duplicate movie_ids: 0
revenue nulls: 0, revenue == 0: 0
budget <= 0 or null: 0, runtime <= 0 or null: 0
rows with valid target (revenue > 0): 8040


(8040, 43)

## 2. Leakage-safe historical features

In [5]:
hist = load_history_data()
hist.shape, hist['release_date'].min(), hist['release_date'].max()

((14895, 8),
 Timestamp('1902-06-15 00:00:00'),
 Timestamp('2029-12-19 00:00:00'))

In [6]:
base = prepare_history_table(hist)
print("history universe (budget > 0 and revenue > 0):", base.shape[0])
base[['movie_id', 'release_date', 'budget', 'revenue', 'director_ids', 'company_ids', 'cast_ids']].head()

history universe (budget > 0 and revenue > 0): 8449


,movie_id,release_date,budget,revenue,director_ids,company_ids,cast_ids
0,5,1995-12-09,4000000,4257354,"[138, 2294, 3110, 3111]","[14, 59]","[3129, 3130, 2555]"
1,6,1993-10-15,21000000,12136938,[2042],"[33, 182, 1644]","[2880, 9777, 5724]"
2,11,1977-05-25,11000000,775398007,[1],[1],"[2, 3, 4]"
3,12,2003-05-30,94000000,940335536,[7],[3],"[13, 14, 12]"
4,13,1994-06-23,55000000,677387716,[24],"[4, 412, 21920]","[31, 32, 33]"


For each film we compute aggregates over **earlier** releases of:

- its directors (movie count, avg revenue/budget/ROI, max revenue)
- its production companies (avg revenue)
- its top-3 cast (avg revenue)
- its collection / franchise (movie count, avg revenue, avg ROI)

The naive-looking rule is enforced: `release_date` of every contributing movie must be strictly smaller than the film's own release date.

In [7]:
hist_features = compute_historical_features(hist)
hist_features.shape

(8449, 12)

### Leak check — recompute prior features brute-force for a random sample

In [8]:
verify_historical_features(hist, hist_features, n=25)

director_previous_avg_revenue: max relative difference vs brute force = 0.0000
production_company_previous_avg_revenue: max relative difference vs brute force = 0.0000


cast_previous_avg_revenue: max relative difference vs brute force = 0.0000
lead_actor_previous_avg_revenue: max relative difference vs brute force = 0.0000
verify_historical_features: passed


### Manual sanity check on one film

`Avengers: Infinity War` (2018). Every one of its historical features must come from movies released before 2018; no 2018-or-later film of its directors/actors can be used.

In [9]:
mid = df[df['original_title'] == 'Avengers: Infinity War']['movie_id'].iloc[0]
mrow = base[base['movie_id'] == mid].iloc[0]
print("film release year:", mrow['release_date'].year)
print("directors:", mrow['director_ids'])
print("managed by collections:", mrow['belongs_to_collection_id'])

s = hist_features[hist_features['movie_id'] == mid]
print(s.to_string(index=False))

# confirm the max release year of any prior film by these directors is < 2018
dirs = mrow['director_ids']
prior_years = base[(base['release_date'] < mrow['release_date']) & (base['director_ids'].apply(lambda x: bool(set(x) & set(dirs))))]['release_date'].dt.year
print("max release year among director prior films:", prior_years.max(), "(must be < 2018)")

film release year: 2018
directors: [19271, 19272]
managed by collections: 86311
 movie_id  director_previous_movie_count  director_previous_avg_revenue  director_previous_avg_budget  director_previous_avg_roi  director_previous_max_revenue  production_company_previous_avg_revenue  cast_previous_avg_revenue  lead_actor_previous_avg_revenue  franchise_previous_avg_revenue  franchise_previous_avg_roi  franchise_movie_count
   299536                            8.0                   501211203.25                   120250000.0                   2.968184                   2004844813.0                             7.410130e+08               3.293348e+08                     4.697590e+08                    1462109604.5                    6.442074                    2.0
max release year among director prior films: 2016 (must be < 2018)


## 3. Build the full feature table

In [10]:
ds = prepare_ml_dataset()
ds.shape

(8040, 66)

In [11]:
ds[['release_quarter', 'release_decade', 'log_budget', 'log_revenue']].head()

,release_quarter,release_decade,log_budget,log_revenue
0,4,1990,15.201805,15.264159
1,4,1990,16.860033,16.311764
2,2,1970,16.213406,20.468887
3,2,2000,18.358805,20.661747
4,2,1990,17.822844,20.333754


## 4. Missing values in the historical block

In [12]:
hist_cols = [
    'director_previous_movie_count', 'director_previous_avg_revenue',
    'director_previous_avg_budget', 'director_previous_avg_roi',
    'director_previous_max_revenue', 'production_company_previous_avg_revenue',
    'cast_previous_avg_revenue', 'lead_actor_previous_avg_revenue',
    'franchise_previous_avg_revenue',
    'franchise_previous_avg_roi', 'franchise_movie_count',
]
ms = ds[hist_cols].isna().mean().round(3)
pd.DataFrame({'missing_share': ms})

,missing_share
director_previous_movie_count,0.001
director_previous_avg_revenue,0.391
director_previous_avg_budget,0.391
director_previous_avg_roi,0.391
director_previous_max_revenue,0.391
production_company_previous_avg_revenue,0.087
cast_previous_avg_revenue,0.126
lead_actor_previous_avg_revenue,0.377
franchise_previous_avg_revenue,0.873
franchise_previous_avg_roi,0.873


Films with no prior releases for their director / company / cast keep `NaN` in those columns.
Tree models (XGBoost / LightGBM / RandomForest) handle this natively; the linear models median-impute on the training split only.

## 5. Save the processed table for the modelling notebooks

In [13]:
out = PROJECT_ROOT / 'data' / 'processed' / 'ml_features.csv'
out.parent.mkdir(parents=True, exist_ok=True)
ds.to_csv(out, index=False)
print("saved", out)

saved /Users/hariz/Desktop/TMDB-movie-analysis/data/processed/ml_features.csv
